# Projet d'analyse de corpus — Prétraitement de *Le Féminisme* d'Émile Faguet

**Objectif : étudier la dépendance des traitements lexicaux au nettoyage préalable du corpus.**

Une mesure lexicale repose sur les unités reconnues dans le texte. La préparation de ces unités (délimitation, normalisation, traitement des éléments éditoriaux) conditionne donc les résultats obtenus. Voir Manning, Raghavan et Schütze (2008, chap. 2) et Bird, Klein et Loper (2009, chap. 3).

**Déroulement :** découvrir les fonctions ; produire et sauvegarder une première version nettoyée ; inspecter le document et rechercher d'éventuelles anomalies ; estimer leur fréquence et décider de poursuivre ou de tester une révision ; comparer enfin les mesures lexicales.

Les sources sont communes : texte original figé, fonctions Python issues d'une révision Git identifiée et profil de nettoyage partagé. Vos observations et décisions restent personnelles.

Enregistrez une copie de ce notebook dans **votre Drive**. En fin de séance, déposez les résultats dans le **dossier partagé**.

*Références :* Manning, C. D., Raghavan, P. et Schütze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press, chapitre 2, « The term vocabulary and postings lists ». https://doi.org/10.1017/CBO9780511809071 ; Bird, S., Klein, E. et Loper, E. (2009). *Natural Language Processing with Python*. O'Reilly Media, chapitre 3. https://www.nltk.org/book/ch03.html.

## Initialisation — Versions et sources

La version du notebook et la révision Git de la bibliothèque permettent de retrouver les fonctions utilisées pour l'analyse. Après une correction, créez une copie du notebook et changez simplement ces paramètres.

Tous les fichiers produits pendant la séance seront regroupés dans le dossier de travail de Colab, par défaut **`/content/analyse_faguet_v2_0_0`**.

In [ ]:
import hashlib
import secrets
from datetime import datetime, timezone
import importlib
import json
from pathlib import Path
import subprocess
import sys
from urllib.parse import quote
from urllib.request import urlopen

DEPOT_LIB = "dreymond732/tal-notebook-evaluator"
SOUS_DOSSIER_LIB = "PT-LLMvsPy"

# Versions utilisées pour cette analyse.
VERSION_NOTEBOOK = "2.0.0"
REVISION_GIT = "main"
DOSSIER_DRIVE_ID = "18KPGO5jGPt5f8t_Gq1km79hno4rqy1xO"
URL_DOSSIER_DRIVE = "https://drive.google.com/drive/folders/" + DOSSIER_DRIVE_ID
DOSSIER_TRAVAIL = Path("/content") / (
    "analyse_faguet_v" + VERSION_NOTEBOOK.replace(".", "_")
)
if not Path("/content").is_dir():
    DOSSIER_TRAVAIL = Path.cwd() / DOSSIER_TRAVAIL.name
DOSSIER_TRAVAIL.mkdir(parents=True, exist_ok=True)

# Une étiquette automatique par exécution, sans identifiant personnel à saisir.
NUMERO_SESSION = f"{secrets.randbelow(1_000_000_000):09d}"
DATE_SESSION = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

try:
    import google.colab
    EST_COLAB = True
except ImportError:
    EST_COLAB = False

print("Notebook :", VERSION_NOTEBOOK)
def figer_revision_git(depot, reference):
    """Résout une branche ou un tag en commit immuable pour toute la session."""
    url = ("https://api.github.com/repos/" + depot
           + "/commits/" + quote(reference, safe=""))
    with urlopen(url, timeout=30) as reponse:
        commit = json.loads(reponse.read().decode("utf-8"))["sha"]
    if len(commit) != 40 or any(c not in "0123456789abcdef" for c in commit):
        raise ValueError("Révision Git incorrecte")
    return commit

REVISION_GIT_RESOLUE = figer_revision_git(DEPOT_LIB, REVISION_GIT)
print("Révision demandée :", REVISION_GIT)
print("Commit des fonctions :", REVISION_GIT_RESOLUE)
print("Dossier des résultats :", DOSSIER_TRAVAIL)
print("Session :", NUMERO_SESSION, "| date :", DATE_SESSION)

In [ ]:
def installer_bibliotheque_depuis_git():
    for racine in (Path.cwd(), Path.cwd().parent):
        if (racine / "src" / "tal_toolkit" / "inspection_td01.py").exists():
            sys.path.insert(0, str(racine / "src"))
            print("Bibliothèque chargée depuis :", racine)
            return "clone-local"
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--force-reinstall", "--no-deps",
        f"git+https://github.com/{DEPOT_LIB}.git@{REVISION_GIT_RESOLUE}"
        f"#subdirectory={SOUS_DOSSIER_LIB}",
    ])
    importlib.invalidate_caches()
    return REVISION_GIT_RESOLUE

REFERENCE_BIBLIOTHEQUE = installer_bibliotheque_depuis_git()

try:
    import spacy
    from packaging.version import Version
    if Version(spacy.__version__) < Version("3.8"):
        raise ImportError("spaCy ancien")
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "spacy>=3.8,<4"
    ])
    import spacy

from tal_toolkit import (
    __version__ as VERSION_BIBLIOTHEQUE, charger_corpus,
    OperationPretraitement, PlanPretraitement,
    executer_pretraitements, recenser_elements, composer_texte_epure,
    comparer_corpus_td01,

    plan_standard_td01,
    verifier_corpus_standard_td01,
    SHA_FAGUET_TD01, SHA_FAGUET_EPURE_TD01, METRIQUES_TD01,
    localiser_dans_original, fonctions_ayant_modifie_zone,
    compter_motif,
)
assert tuple(int(x) for x in VERSION_BIBLIOTHEQUE.split(".")[:2]) >= (0, 3)
print("Bibliothèque :", VERSION_BIBLIOTHEQUE, "| spaCy :", spacy.__version__)

### Charger le texte original et le document Gemini

Le texte source est conservé pour retrouver ultérieurement toute zone concernée par le nettoyage. Le document Gemini fournit les mesures à mettre en regard des calculs.

In [ ]:
COMMIT_SOURCE = "b92e92b6c216014283056c83aa545881a447b3a5"
CHEMIN_SOURCE = "Notebooks TD/S3/ressources/faguet_source.txt"
SHA256_SOURCE = "c9832aecd99b11f3c769d11173fd6039a4948a8ee86484658dd8c7a25848a8fa"

fichier_original = DOSSIER_TRAVAIL / "faguet_source.txt"
if not fichier_original.exists():
    url = (
        "https://raw.githubusercontent.com/dreymond732/tal-notebook-evaluator/"
        + COMMIT_SOURCE + "/" + quote(CHEMIN_SOURCE, safe="/")
    )
    with urlopen(url, timeout=60) as reponse:
        fichier_original.write_bytes(reponse.read())

corpus_source = charger_corpus(fichier_original, sha256_attendu=SHA256_SOURCE)
verifier_corpus_standard_td01(corpus_source)
print("Corpus chargé :", len(corpus_source.texte), "caractères")

In [ ]:
URL_GEMINI = (
    "https://raw.githubusercontent.com/" + DEPOT_LIB
    + "/" + REVISION_GIT_RESOLUE + "/" + SOUS_DOSSIER_LIB + "/data/annonces_globales_gemini.json"
)
with urlopen(URL_GEMINI, timeout=30) as reponse:
    ANNONCES_GEMINI = json.loads(reponse.read().decode("utf-8"))
assert ANNONCES_GEMINI["version"] == "gemini_metrics_globales_v1"
for item in ANNONCES_GEMINI["annonces"]:
    print(item["libelle"], ":", item["valeur"])

## 1 — Fonctions disponibles

Le nettoyage peut porter sur les **notes et renvois éditoriaux**, les **séparateurs visuels**, les **retours de ligne typographiques** ou certains caractères. Les titres sont conservés.

| Fonction | Rôle |
|---|---|
| `recenser_elements()` | Repérer les éléments présents dans le texte |
| `executer_pretraitements()` | Appliquer le plan et compter les modifications |
| `composer_texte_epure()` | Construire le texte lisible après masquage et mise en forme |
| `exporter_texte_approuve()` | Enregistrer une version retenue et ses informations de provenance |

Le nettoyage du texte, le découpage en tokens et le choix des tokens à compter sont des opérations distinctes.

In [ ]:
inventaire = recenser_elements(corpus_source, max_exemples=12)
print("Retours CRLF :", inventaire["retours_crlf"])
for nom, donnees in inventaire["categories"].items():
    print("\n", nom, ":", donnees["nombre"])
    for exemple in donnees["exemples"][:5]:
        print("  ", exemple["debut"], repr(exemple["texte"][:100]))

def operation(nom, justification, cibles=()):
    return OperationPretraitement(nom, justification, tuple(cibles))

### Illustrations des transformations

Les exemples suivants montrent comment les fonctions réagissent aux renvois de notes, aux coupures de ligne, aux séparateurs, aux titres et aux césures. Ils constituent des tests unitaires sur des textes courts.

In [ ]:
EXEMPLES_FOURNIS = [
    {
        "nom": "retours typographiques",
        "entree": "Une phrase\r\ncontinue.\r\n\r\nAutre paragraphe.",
        "ops": [operation("recomposer_lignes", "Réunir les lignes typographiques")],
        "attendu": "Une phrase continue.\n\nAutre paragraphe.",
    },
    {
        "nom": "apostrophes courbes",
        "entree": "L’homme et l’histoire.",
        "ops": [operation("apostrophes_droites", "Normaliser les apostrophes")],
        "attendu": "L'homme et l'histoire.",
    },
    {
        "nom": "appel et contenu de note",
        "entree": "Un propos[1].\r\n\r\n[Note 1: Une référence.]\r\n\r\nLa suite.",
        "ops": [operation("masquer_appels_notes", "Masquer l'appel"),
                operation("masquer_blocs_notes", "Masquer le bloc")],
        "attendu": "Un propos.\n\nLa suite.",
    },
    {
        "nom": "séparateur d'astérisques",
        "entree": "Texte un.\r\n\r\n* * * * *\r\n\r\nTexte deux.",
        "ops": [operation("masquer_separateurs", "Masquer le séparateur")],
        "attendu": "Texte un.\n\nTexte deux.",
    },
    {
        "nom": "titre conservé",
        "entree": "IDÉES GÉNÉRALES\r\n\r\nLe chapitre commence.",
        "ops": [],
        "attendu": "IDÉES GÉNÉRALES\n\nLe chapitre commence.",
    },
    {
        "nom": "césure approuvée",
        "entree": "Une infor-\r\nmation importante.",
        "ops": [operation(
            "raccorder_cesures_validees", "Césure validée sur cet exemple",
            ("infor-\r\nmation",),
        )],
        "attendu": "Une information importante.",
    },
]
for cas in EXEMPLES_FOURNIS:
    plan_demo = PlanPretraitement(tuple(cas["ops"]), identifiant="demo-v1")
    test = executer_pretraitements(cas["entree"], plan=plan_demo)
    observe = composer_texte_epure(test).texte
    assert observe == cas["attendu"], cas["nom"]
    assert test.original == cas["entree"]
    assert test.indices_conserves
    print("OK :", cas["nom"], "=>", repr(observe))

## 2 — Nettoyer le corpus complet

Le nettoyage retire les notes et les séparateurs éditoriaux, puis recompose les lignes typographiques. Le texte obtenu est enregistré dans le dossier de travail, **sans qu'une décision de validation soit exigée**.

Ouvrez le fichier produit dans Bloc-notes ou un éditeur de texte simple. Observez le début, le milieu et la fin du document ainsi que quelques passages choisis : paragraphes, titres, ponctuation, mots mal raccordés et résidus éventuels.

In [ ]:
plan_global = plan_standard_td01()
resultat_global = executer_pretraitements(
    corpus_source, plan=plan_global, max_exemples=40
)
vue_epuree = composer_texte_epure(resultat_global)

assert resultat_global.sha256_original == SHA256_SOURCE
assert len(vue_epuree.indices_source) == len(vue_epuree.texte)

FICHIER_APERCU = DOSSIER_TRAVAIL / "faguet_nettoye.txt"
with FICHIER_APERCU.open("w", encoding="utf-8", newline="") as f:
    f.write(vue_epuree.texte)

print("Texte nettoyé :", FICHIER_APERCU)
print("Caractères : original", len(corpus_source.texte),
      "| nettoyé", len(vue_epuree.texte))
for element in resultat_global.execution:
    print(element["operation"], ":", element["modifications"], "modifications")

In [ ]:
# Dans Colab : téléchargez le fichier texte, puis ouvrez-le avec Bloc-notes.
if EST_COLAB:
    from google.colab import files
    files.download(str(FICHIER_APERCU))
else:
    print("Ouvrez ce fichier dans votre éditeur :", FICHIER_APERCU)

## 3 — Rechercher et documenter les anomalies

Après lecture du fichier nettoyé, si vous suspectez une erreur, procédez de façon précise :

1. Choisissez dans **le texte original** une courte suite de caractères caractéristique du problème (l'**ancre**).
2. Retrouvez sa position et vérifiez la fenêtre de texte autour d'elle. Si l'ancre apparaît plusieurs fois, sélectionnez la bonne occurrence.
3. Examinez quelles fonctions ont modifié cette zone. Il s'agit de **pistes de diagnostic**, à confronter au texte.
4. Choisissez un motif représentatif et comptez ses occurrences dans le texte original et le texte nettoyé. Un motif littéral dénombre exactement cette suite de caractères ; une expression régulière peut couvrir plusieurs variantes. **Ce comptage estime la portée du motif, pas nécessairement celle de toutes les erreurs apparentées.**

Vous pouvez documenter plusieurs anomalies. Si vous n'en trouvez aucune, décrivez les parties du document réellement inspectées : un résultat sans anomalie reste une observation utile.

In [ ]:
# EXEMPLE de recherche : remplacer la chaîne par une ancre de votre choix.
ANCRE_A_CHERCHER = ""         # Suite exacte présente dans le texte original
NUMERO_OCCURRENCE = 1         # 1 pour la première occurrence, 2 pour la deuxième, etc.
RAYON_CONTEXTE = 120

if ANCRE_A_CHERCHER:
    zone = localiser_dans_original(
        corpus_source.texte, ANCRE_A_CHERCHER,
        occurrence=NUMERO_OCCURRENCE, contexte=RAYON_CONTEXTE,
    )
    print("Nombre de correspondances :", zone["nombre_correspondances"])
    print("Bornes vérifiées :", zone["fenetre_debut"], zone["fenetre_fin"])
    print("Fenêtre originale :", repr(zone["fenetre_originale"]))
    print("Vue de contrôle :", repr(resultat_global.analyse[
        zone["fenetre_debut"]:zone["fenetre_fin"]
    ]))
    print("Opérations intervenues dans la fenêtre :")
    for intervention in fonctions_ayant_modifie_zone(
        resultat_global, zone["fenetre_debut"], zone["fenetre_fin"]
    ):
        print(" ", intervention)
else:
    print("Renseignez ANCRE_A_CHERCHER pour examiner une zone du texte.")

In [ ]:
# Estimer les occurrences d'un motif lié au problème observé.
MOTIF_A_COMPTER = ""     # Mot, caractères résiduels ou expression régulière
MODE_MOTIF = "litteral"  # "litteral" ou "regex"

if MOTIF_A_COMPTER:
    resultat_occurrences = compter_motif(
        corpus_source.texte, vue_epuree.texte, MOTIF_A_COMPTER, mode=MODE_MOTIF
    )
    print("Occurrences originales :", resultat_occurrences["occurrences_original"])
    print("Occurrences après nettoyage :", resultat_occurrences["occurrences_epure"])
    print("Écart :", resultat_occurrences["difference"])
    print(resultat_occurrences["limite"])
else:
    print("Renseignez MOTIF_A_COMPTER pour évaluer la fréquence d'un cas.")

### Noter les observations éventuelles

Les deux cellules précédentes permettent de retrouver une chaîne dans le texte original et de compter les occurrences d'un motif avant et après nettoyage.

Vous pouvez consigner librement les anomalies ou les particularités repérées dans **`NOTES_INSPECTION`**, par exemple la chaîne recherchée, la fonction concernée, les occurrences et leur conséquence possible. Aucune anomalie n'est imposée : vous pouvez aussi décrire les portions du texte examinées.

Ces notes seront jointes aux résultats, mais leur présence n'est pas une condition pour lancer les calculs.

In [ ]:
# Facultatif : notes libres sur les passages examinés et les défauts éventuels.
NOTES_INSPECTION = """
"""

FICHIER_INSPECTION = DOSSIER_TRAVAIL / "observations_nettoyage.txt"
FICHIER_INSPECTION.write_text(
    NOTES_INSPECTION.strip() + "\n", encoding="utf-8"
)
print("Observations enregistrées :", FICHIER_INSPECTION)

### Passer aux mesures lexicales

Poursuivez directement les calculs sur le texte qui vient d'être produit. Un éventuel défaut de nettoyage pourra être discuté à partir du fichier, des observations et des métriques obtenues. Vous pourrez ultérieurement comparer une nouvelle révision de la bibliothèque avec cette première version.

In [ ]:
# Le fichier obtenu dans la section 2 sert directement aux calculs.
assert FICHIER_APERCU.is_file(), "Exécutez d'abord le nettoyage complet."
print("Corpus utilisé :", FICHIER_APERCU)
print("Les mesures peuvent commencer.")

## 4 — Calculer et comparer les mesures lexicales

Le texte original et le texte nettoyé sont analysés avec **les mêmes paramètres spaCy**, sans nouvelle transformation.

Le tableau compare les tokens, les types, les hapax, le TTR, la proportion d'hapax, le nombre de phrases et leur longueur moyenne. Les résultats peuvent être utilisés dès maintenant pour examiner l'effet du prétraitement, même si des améliorations sont envisagées.

In [ ]:
corpus_epure = charger_corpus(
    FICHIER_APERCU, sha256_attendu=vue_epuree.sha256_texte_utf8
)
nlp = spacy.blank("fr")
nlp.add_pipe("sentencizer")
nlp.max_length = max(nlp.max_length, len(corpus_source.texte) + 1000)

resultat_comparaison = comparer_corpus_td01(
    corpus_source, corpus_epure, nlp, protocole=METRIQUES_TD01
)
assert resultat_comparaison["source_originale_sha256"] == SHA256_SOURCE
assert resultat_comparaison["source_epuree_sha256"] == vue_epuree.sha256_texte_utf8

import pandas as pd
tableau_comparaison = pd.DataFrame(resultat_comparaison["comparaison"])
display(tableau_comparaison)

FICHIER_COMPARAISON = DOSSIER_TRAVAIL / "comparaison_metriques.csv"
tableau_comparaison.to_csv(FICHIER_COMPARAISON, index=False, encoding="utf-8")
FICHIER_COMPARAISON_JSON = DOSSIER_TRAVAIL / "comparaison_metriques.json"
FICHIER_COMPARAISON_JSON.write_text(
    json.dumps(resultat_comparaison, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8"
)
print("Fichiers enregistrés dans :", DOSSIER_TRAVAIL)

In [ ]:
# Informations de provenance nécessaires à la reprise ou à la comparaison.
METADONNEES_ANALYSE = {
    "schema": "analyse-faguet-resultats-v1",
    "numero_session": NUMERO_SESSION,
    "date_utc": DATE_SESSION,
    "version_notebook": VERSION_NOTEBOOK,
    "revision_git": REVISION_GIT_RESOLUE,
    "version_bibliotheque": VERSION_BIBLIOTHEQUE,
    "sha256_source": SHA256_SOURCE,
    "sha256_nettoye": vue_epuree.sha256_texte_utf8,
    "plan_pretraitement": resultat_global.plan,
    "operations": list(resultat_global.execution),
    "protocole_mesures": resultat_comparaison["protocole_commun"],
    "logiciel": resultat_comparaison["logiciel"],
}
FICHIER_METADONNEES = DOSSIER_TRAVAIL / "metadonnees_analyse.json"
FICHIER_METADONNEES.write_text(
    json.dumps(METADONNEES_ANALYSE, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8"
)
print("Provenance et transformations :", FICHIER_METADONNEES)

### Interpréter les différences

Examinez les variations des tokens, types, hapax et phrases. Les observations peuvent être consignées librement, avec un exemple du texte ou une remarque sur les chiffres de Gemini.

Les fichiers produits restent accessibles dans **`/content/analyse_faguet_v2_0_0`** pour la version actuelle du notebook. Ils ne dépendent pas d'un choix « conserver / réviser » ni d'un identifiant personnel.

In [ ]:
# Facultatif : commentaires d'analyse, sans formulaire ni longueur imposée.
COMMENTAIRES_ANALYSE = """
"""
FICHIER_COMMENTAIRES = DOSSIER_TRAVAIL / "commentaires_analyse.txt"
FICHIER_COMMENTAIRES.write_text(
    COMMENTAIRES_ANALYSE.strip() + "\n", encoding="utf-8"
)

# Archive datée et identifiée par un nombre aléatoire.
import zipfile
NOM_ARCHIVE = (
    f"FAGUET_v{VERSION_NOTEBOOK.replace('.', '_')}_"
    f"{DATE_SESSION}_{NUMERO_SESSION}.zip"
)
FICHIER_ARCHIVE = DOSSIER_TRAVAIL / NOM_ARCHIVE
fichiers_resultats = [
    FICHIER_APERCU, FICHIER_INSPECTION,
    FICHIER_COMPARAISON, FICHIER_COMPARAISON_JSON,
    FICHIER_METADONNEES, FICHIER_COMMENTAIRES,
]
with zipfile.ZipFile(FICHIER_ARCHIVE, "w", zipfile.ZIP_DEFLATED) as archive:
    for fichier in fichiers_resultats:
        assert fichier.is_file(), f"Fichier manquant : {fichier}"
        archive.write(fichier, arcname=fichier.name)

with zipfile.ZipFile(FICHIER_ARCHIVE) as archive:
    assert archive.testzip() is None
    print("Archive :", FICHIER_ARCHIVE)
    print("Contenu :", ", ".join(archive.namelist()))
print("Tous les résultats se trouvent dans :", DOSSIER_TRAVAIL)

## Déposer les résultats dans votre Drive

Montez votre Drive dans Colab et recherchez le dossier partagé du projet. S'il apparaît dans « Partagés avec moi », vous pouvez ajouter un **raccourci vers ce dossier dans Mon Drive** pour le retrouver parmi les fichiers montés.

Les résultats restent disponibles dans **`/content/analyse_faguet_v2_0_0`** : vous pouvez copier l'archive ZIP ou consulter ses fichiers directement.

Dossier partagé : https://drive.google.com/drive/folders/18KPGO5jGPt5f8t_Gq1km79hno4rqy1xO

In [ ]:
# Monter le Drive pour accéder aux dossiers visibles dans Mon Drive.
if EST_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MON_DRIVE = Path("/content/drive/MyDrive")
    print("Dossiers visibles dans Mon Drive :")
    for dossier in sorted(MON_DRIVE.iterdir()):
        if dossier.is_dir():
            print("  ", dossier)
else:
    MON_DRIVE = None
    print("Cette étape de montage concerne Google Colab.")

In [ ]:
# Indiquez le chemin du dossier partagé tel qu'il apparaît dans le Drive monté.
# Exemple (à adapter) : "/content/drive/MyDrive/Projet_PT_LLMvsPy"
DOSSIER_DEPOT_DRIVE = ""

if not DOSSIER_DEPOT_DRIVE.strip():
    print("Renseignez DOSSIER_DEPOT_DRIVE après avoir repéré le dossier partagé.")
    print("Archive locale disponible :", FICHIER_ARCHIVE)
else:
    from shutil import copy2
    destination = Path(DOSSIER_DEPOT_DRIVE).expanduser()
    if not destination.is_dir():
        raise FileNotFoundError(
            "Dossier Drive introuvable : vérifiez son chemin et les droits de partage."
        )
    chemin_depot = destination / FICHIER_ARCHIVE.name
    if chemin_depot.exists():
        raise FileExistsError("Ce nom existe déjà : aucun fichier n'est écrasé.")
    copy2(FICHIER_ARCHIVE, chemin_depot)
    sha_local = hashlib.sha256(FICHIER_ARCHIVE.read_bytes()).hexdigest()
    sha_drive = hashlib.sha256(chemin_depot.read_bytes()).hexdigest()
    if sha_drive != sha_local:
        raise IOError("Le fichier copié dans Drive n'a pas la même empreinte.")
    print("Archive déposée :", chemin_depot)
    print("SHA-256 vérifié :", sha_drive)

## Suite du projet

Le dossier de travail contient le texte nettoyé, les résultats lexicaux, les observations libres et les paramètres permettant de reproduire les calculs.

Une correction de la bibliothèque pourra être évaluée en changeant `REVISION_GIT` et `VERSION_NOTEBOOK` dans une copie de ce notebook, puis en comparant les fichiers obtenus.